# Lesson 2.2: Advanced Prompting Techniques

**Duration:** 1 hour  
**Level:** Intermediate  

---

## Learning Objectives

By the end of this lesson, you will:
- Master Chain-of-Thought (CoT) prompting for complex reasoning
- Implement Tree-of-Thought (ToT) for exploring multiple solution paths
- Apply the ReAct pattern for tool-using agents
- Use Self-Consistency for improved accuracy

---

## Why Advanced Techniques?

Basic prompting works for simple tasks, but complex problems require sophisticated approaches:

| Technique | Best For | Key Insight |
|-----------|----------|-------------|
| **Chain-of-Thought** | Math, logic, multi-step reasoning | "Show your work" improves accuracy |
| **Tree-of-Thought** | Complex decisions, creative problems | Explore multiple paths, pick the best |
| **ReAct** | Tool use, real-world tasks | Interleave reasoning and action |
| **Self-Consistency** | High-stakes decisions | Multiple samples, majority vote |

## Setup

In [ ]:
!pip install google-generativeai langchain-google-genai python-dotenv -q

In [ ]:
import os
import json
import re
from typing import List, Dict, Callable
from collections import Counter
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel('gemini-1.5-flash')

print("Setup complete!")

---

## 1. Chain-of-Thought (CoT) Prompting

### What is Chain-of-Thought?

Chain-of-Thought prompting encourages the model to **show its reasoning step by step** before giving a final answer. This dramatically improves performance on:
- Mathematical problems
- Logical reasoning
- Multi-step tasks
- Complex analysis

### Two Types of CoT:
1. **Zero-Shot CoT**: Just add "Let's think step by step"
2. **Few-Shot CoT**: Provide examples showing reasoning

### Industry Example: Financial Analysis

In [ ]:
# Standard prompting vs Chain-of-Thought
financial_problem = """
A company has:
- Q1 Revenue: $2.5M
- Q2 Revenue: $2.8M (includes one-time contract of $500K)
- Operating costs: 60% of revenue
- Q3 projection: 15% organic growth over Q2's recurring revenue

What is the projected Q3 profit?
"""

print("STANDARD vs CHAIN-OF-THOUGHT COMPARISON")
print("="*60)

# Standard prompting
standard_response = model.generate_content(
    financial_problem,
    generation_config=genai.GenerationConfig(temperature=0)
)

print("\n📊 STANDARD PROMPTING:")
print("-"*40)
print(standard_response.text[:500])

# Chain-of-Thought prompting
cot_prompt = financial_problem + "\n\nLet's solve this step by step:"

cot_response = model.generate_content(
    cot_prompt,
    generation_config=genai.GenerationConfig(temperature=0)
)

print("\n🧠 CHAIN-OF-THOUGHT PROMPTING:")
print("-"*40)
print(cot_response.text)

In [ ]:
def zero_shot_cot(question: str) -> Dict:
    """
    Zero-Shot Chain-of-Thought: Just add the magic phrase.
    """
    prompt = f"""{question}

Let's approach this step by step:
1. First, identify the key information
2. Then, work through the logic
3. Finally, provide the answer
"""
    
    response = model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(temperature=0)
    )
    
    return {
        "reasoning": response.text,
        "method": "zero_shot_cot"
    }

# Industry Example: Pricing Decision
pricing_question = """
Our SaaS product costs $15/month per user to operate.
Competitors charge between $25-40/user/month.
We have 1000 users and want 40% gross margin.
Our customer acquisition cost is $120 per user.
Average customer lifetime is 24 months.

What should our price be, and what's our expected LTV:CAC ratio?
"""

result = zero_shot_cot(pricing_question)

print("ZERO-SHOT COT: PRICING ANALYSIS")
print("="*60)
print(result["reasoning"])

In [ ]:
def few_shot_cot(question: str, examples: List[Dict]) -> Dict:
    """
    Few-Shot Chain-of-Thought: Provide reasoning examples.
    """
    # Build examples section
    examples_text = []
    for ex in examples:
        examples_text.append(f"""Problem: {ex['question']}

Reasoning:
{ex['reasoning']}

Answer: {ex['answer']}""")
    
    examples_str = "\n\n---\n\n".join(examples_text)
    
    prompt = f"""Here are some examples of how to solve problems step by step:

{examples_str}

---

Now solve this problem using the same approach:

Problem: {question}

Reasoning:"""
    
    response = model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(temperature=0)
    )
    
    return {
        "reasoning": response.text,
        "method": "few_shot_cot"
    }

# Few-shot examples for business metrics
metric_examples = [
    {
        "question": "A company has 100 customers paying $50/month. 5 customers churned this month. What's the MRR and churn rate?",
        "reasoning": """1. Calculate initial MRR: 100 customers × $50 = $5,000 MRR
2. Calculate churned revenue: 5 customers × $50 = $250
3. Calculate new MRR: $5,000 - $250 = $4,750
4. Calculate churn rate: 5 ÷ 100 = 5% monthly churn""",
        "answer": "MRR is $4,750 with a 5% monthly churn rate."
    },
    {
        "question": "CAC is $200, average revenue per user is $40/month, and customers stay 18 months on average. What's the LTV and LTV:CAC?",
        "reasoning": """1. Calculate LTV: $40/month × 18 months = $720
2. Calculate LTV:CAC ratio: $720 ÷ $200 = 3.6
3. Evaluate: Ratio > 3 is generally healthy for SaaS""",
        "answer": "LTV is $720, LTV:CAC ratio is 3.6 (healthy)."
    }
]

new_problem = """
Our startup has:
- 500 paying customers
- Average revenue per user: $80/month
- Monthly churn: 3%
- Marketing spend: $50,000/month
- New customers acquired: 75/month

Calculate CAC, expected customer lifetime, LTV, and whether this is sustainable.
"""

result = few_shot_cot(new_problem, metric_examples)

print("FEW-SHOT COT: SaaS METRICS ANALYSIS")
print("="*60)
print(result["reasoning"])

In [ ]:
# LangChain implementation of CoT
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

# CoT prompt template
cot_template = ChatPromptTemplate.from_messages([
    ("system", """You are an expert business analyst. When solving problems:
1. Break down the problem into components
2. Show your calculations clearly
3. Explain your reasoning at each step
4. State assumptions explicitly
5. Provide a clear final answer"""),
    ("human", "{problem}\n\nLet's analyze this step by step:")
])

cot_chain = cot_template | llm | StrOutputParser()

# Complex business problem
problem = """
We're considering expanding our engineering team.
Current team: 10 engineers, velocity: 100 story points/sprint
Each new hire costs $150K/year fully loaded
Ramp-up time: 3 months to full productivity
During ramp-up: 25% productivity
Senior engineers spend 20% time mentoring new hires

If we hire 5 new engineers, what's our expected velocity after 6 months,
and what's the total cost for this period?
"""

result = cot_chain.invoke({"problem": problem})

print("LANGCHAIN COT: TEAM SCALING ANALYSIS")
print("="*60)
print(result)

---

## 2. Tree-of-Thought (ToT)

### What is Tree-of-Thought?

Tree-of-Thought extends CoT by **exploring multiple reasoning paths** and using the model to evaluate which paths are most promising.

**Process:**
1. **Generate** multiple initial approaches
2. **Evaluate** each approach's promise
3. **Prune** unpromising branches
4. **Expand** the best approaches

### Industry Example: System Architecture Decision

In [ ]:
class TreeOfThought:
    """
    Tree-of-Thought implementation for complex decision making.
    """
    
    def __init__(self):
        self.model = genai.GenerativeModel('gemini-1.5-flash')
    
    def generate_approaches(self, problem: str, num_approaches: int = 3) -> List[str]:
        """Generate multiple initial approaches to the problem."""
        prompt = f"""Problem: {problem}

Generate {num_approaches} distinct approaches to solve this problem.
For each approach:
1. Give it a descriptive name
2. Explain the key idea in 2-3 sentences
3. List main pros and cons

Format:
APPROACH 1: [Name]
Idea: [Description]
Pros: [List]
Cons: [List]

APPROACH 2: [Name]
...
"""
        
        response = self.model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(temperature=0.7)
        )
        
        # Parse approaches
        text = response.text
        approaches = []
        
        pattern = r'APPROACH \d+:(.+?)(?=APPROACH \d+:|$)'
        matches = re.findall(pattern, text, re.DOTALL)
        
        return [m.strip() for m in matches] if matches else [text]
    
    def evaluate_approach(self, problem: str, approach: str) -> Dict:
        """Evaluate how promising an approach is."""
        prompt = f"""Problem: {problem}

Proposed Approach:
{approach}

Evaluate this approach on a scale of 1-10 for each criterion:
1. Feasibility: How practical is this to implement?
2. Effectiveness: How well does this solve the core problem?
3. Risk: How risky is this approach? (10 = low risk)
4. Scalability: How well will this scale?
5. Cost-Efficiency: How cost-effective is this?

Return JSON:
{{
    "feasibility": {"score": X, "reason": "..."},
    "effectiveness": {"score": X, "reason": "..."},
    "risk": {"score": X, "reason": "..."},
    "scalability": {"score": X, "reason": "..."},
    "cost_efficiency": {"score": X, "reason": "..."},
    "overall_score": X,
    "recommendation": "proceed" | "consider" | "avoid"
}}
"""
        
        response = self.model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(
                temperature=0,
                response_mime_type="application/json"
            )
        )
        
        try:
            return json.loads(response.text)
        except:
            return {"overall_score": 5, "recommendation": "consider"}
    
    def expand_approach(self, problem: str, approach: str) -> str:
        """Expand a promising approach into a detailed plan."""
        prompt = f"""Problem: {problem}

Selected Approach:
{approach}

Develop this approach into a detailed implementation plan:
1. Step-by-step implementation roadmap
2. Required resources and timeline
3. Key milestones and success metrics
4. Risk mitigation strategies
5. Expected outcomes
"""
        
        response = self.model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(temperature=0.3)
        )
        
        return response.text
    
    def solve(self, problem: str, num_approaches: int = 3) -> Dict:
        """Full Tree-of-Thought solving process."""
        print("🌳 Starting Tree-of-Thought analysis...\n")
        
        # Step 1: Generate approaches
        print("Step 1: Generating approaches...")
        approaches = self.generate_approaches(problem, num_approaches)
        print(f"   Generated {len(approaches)} approaches\n")
        
        # Step 2: Evaluate each approach
        print("Step 2: Evaluating approaches...")
        evaluations = []
        for i, approach in enumerate(approaches):
            eval_result = self.evaluate_approach(problem, approach)
            evaluations.append({
                "approach": approach,
                "evaluation": eval_result
            })
            print(f"   Approach {i+1}: Score {eval_result.get('overall_score', 'N/A')} - {eval_result.get('recommendation', 'N/A')}")
        
        # Step 3: Select best approach
        print("\nStep 3: Selecting best approach...")
        best = max(evaluations, key=lambda x: x['evaluation'].get('overall_score', 0))
        
        # Step 4: Expand best approach
        print("Step 4: Developing detailed plan...\n")
        detailed_plan = self.expand_approach(problem, best['approach'])
        
        return {
            "all_approaches": evaluations,
            "selected_approach": best,
            "detailed_plan": detailed_plan
        }

In [ ]:
# Use ToT for a complex architecture decision
tot = TreeOfThought()

architecture_problem = """
We need to design a system for a food delivery startup that:
- Handles 10,000 concurrent users during peak hours
- Processes real-time order tracking
- Integrates with 500+ restaurants
- Must be cost-effective (startup budget)
- Needs to support mobile apps (iOS, Android) and web
- Requires 99.9% uptime for payment processing

What architecture approach should we use?
"""

print("TREE-OF-THOUGHT: ARCHITECTURE DECISION")
print("="*60)

result = tot.solve(architecture_problem, num_approaches=3)

print("\n" + "="*60)
print("SELECTED APPROACH:")
print("="*60)
print(result['selected_approach']['approach'][:500])

print("\n" + "="*60)
print("DETAILED IMPLEMENTATION PLAN:")
print("="*60)
print(result['detailed_plan'])

---

## 3. ReAct Pattern (Reasoning + Acting)

### What is ReAct?

ReAct combines **reasoning** with **acting** in an interleaved loop:

```
Thought → Action → Observation → Thought → Action → ... → Answer
```

This is the foundation for tool-using AI agents.

### Industry Example: Customer Data Research Agent

In [ ]:
class ReActAgent:
    """
    ReAct pattern implementation for tool-using agents.
    """
    
    def __init__(self, tools: Dict[str, Callable]):
        self.tools = tools
        self.model = genai.GenerativeModel('gemini-1.5-flash')
        self.trace = []
    
    def _get_tool_descriptions(self) -> str:
        """Format tool descriptions for the prompt."""
        descriptions = []
        for name, func in self.tools.items():
            doc = func.__doc__ or "No description"
            descriptions.append(f"- {name}: {doc.strip()}")
        return "\n".join(descriptions)
    
    def run(self, question: str, max_steps: int = 5) -> Dict:
        """Run the ReAct loop to answer a question."""
        self.trace = []
        
        system_prompt = f"""You are a helpful assistant that answers questions by reasoning and using tools.

Available tools:
{self._get_tool_descriptions()}

For each step, output EXACTLY ONE of these:
1. Thought: [your reasoning about what to do next]
2. Action: [tool_name] Input: [tool input]
3. Answer: [final answer when you have enough information]

IMPORTANT:
- Always start with a Thought
- After an Observation, continue with another Thought
- Only use Answer when you have gathered enough information
- Be concise in your thoughts and actions
"""
        
        context = f"Question: {question}\n\n"
        
        for step in range(max_steps):
            response = self.model.generate_content(
                system_prompt + "\n" + context,
                generation_config=genai.GenerationConfig(temperature=0)
            )
            
            output = response.text.strip()
            self.trace.append({"step": step + 1, "output": output})
            context += output + "\n"
            
            # Check for final answer
            if "Answer:" in output:
                answer = output.split("Answer:")[-1].strip()
                return {
                    "question": question,
                    "answer": answer,
                    "trace": self.trace,
                    "steps": step + 1
                }
            
            # Execute action if present
            if "Action:" in output:
                action_part = output.split("Action:")[-1].strip()
                
                # Parse tool and input
                if "Input:" in action_part:
                    parts = action_part.split("Input:")
                    tool_name = parts[0].strip()
                    tool_input = parts[1].strip()
                else:
                    tool_name = action_part.split()[0]
                    tool_input = ""
                
                # Execute tool
                if tool_name in self.tools:
                    try:
                        observation = self.tools[tool_name](tool_input)
                    except Exception as e:
                        observation = f"Error: {str(e)}"
                else:
                    observation = f"Error: Unknown tool '{tool_name}'"
                
                context += f"\nObservation: {observation}\n\n"
                self.trace.append({"step": step + 1, "observation": observation})
        
        return {
            "question": question,
            "answer": "Could not determine answer within step limit",
            "trace": self.trace,
            "steps": max_steps
        }

In [ ]:
# Define tools for the agent

def search_customers(query: str) -> str:
    """Search customer database by name or email."""
    # Simulated customer database
    customers = {
        "john@acme.com": {"name": "John Smith", "company": "Acme Corp", "plan": "Enterprise", "mrr": 5000},
        "sarah@startup.io": {"name": "Sarah Johnson", "company": "StartupIO", "plan": "Growth", "mrr": 500},
        "mike@bigco.com": {"name": "Mike Chen", "company": "BigCo Inc", "plan": "Enterprise", "mrr": 12000},
    }
    
    for email, data in customers.items():
        if query.lower() in email.lower() or query.lower() in data["name"].lower():
            return json.dumps(data)
    return "No customer found"

def get_usage_stats(customer_email: str) -> str:
    """Get usage statistics for a customer."""
    stats = {
        "john@acme.com": {"api_calls": 150000, "storage_gb": 45, "active_users": 120, "last_active": "2024-01-18"},
        "sarah@startup.io": {"api_calls": 5000, "storage_gb": 2, "active_users": 8, "last_active": "2024-01-15"},
        "mike@bigco.com": {"api_calls": 500000, "storage_gb": 200, "active_users": 500, "last_active": "2024-01-19"},
    }
    
    if customer_email in stats:
        return json.dumps(stats[customer_email])
    return "No usage data found"

def calculate_expansion_potential(mrr: str) -> str:
    """Calculate upsell potential based on current MRR."""
    try:
        mrr_val = int(mrr)
        if mrr_val < 1000:
            potential = "High - candidate for Growth plan upgrade"
        elif mrr_val < 10000:
            potential = "Medium - candidate for Enterprise plan"
        else:
            potential = "Focus on retention and add-ons"
        return potential
    except:
        return "Unable to calculate"

def get_support_history(customer_email: str) -> str:
    """Get support ticket history for a customer."""
    history = {
        "john@acme.com": [{"date": "2024-01-10", "issue": "API rate limiting", "resolved": True}],
        "mike@bigco.com": [{"date": "2024-01-05", "issue": "SSO integration", "resolved": True}, 
                           {"date": "2024-01-18", "issue": "Performance concerns", "resolved": False}],
    }
    return json.dumps(history.get(customer_email, []))

# Create the agent
tools = {
    "search_customers": search_customers,
    "get_usage_stats": get_usage_stats,
    "calculate_expansion_potential": calculate_expansion_potential,
    "get_support_history": get_support_history
}

agent = ReActAgent(tools)

In [ ]:
# Run the agent on a research task
question = """
I need a summary of our customer Mike from BigCo:
- What plan are they on and how much do they pay?
- What's their usage like?
- Any open support issues?
- What's their expansion potential?
"""

print("REACT AGENT: CUSTOMER RESEARCH")
print("="*60)

result = agent.run(question)

print("\n📋 AGENT TRACE:")
print("-"*40)
for item in result['trace']:
    print(f"\nStep {item['step']}:")
    if 'output' in item:
        print(f"  {item['output'][:200]}..." if len(item.get('output', '')) > 200 else f"  {item.get('output', '')}")
    if 'observation' in item:
        print(f"  Observation: {item['observation']}")

print("\n" + "="*60)
print("FINAL ANSWER:")
print("="*60)
print(result['answer'])

In [ ]:
# LangChain ReAct with tools
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.tools import tool
from langchain import hub

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

# Define tools using decorator
@tool
def calculate_metric(expression: str) -> str:
    """Calculate a business metric. Input should be a math expression like '5000 * 12' or '(100 - 5) / 100'."""
    try:
        # Safe evaluation
        allowed = set('0123456789+-*/.() ')
        if all(c in allowed for c in expression):
            result = eval(expression)
            return f"Result: {result}"
        return "Invalid expression"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def lookup_benchmark(metric_name: str) -> str:
    """Look up industry benchmark for a SaaS metric. Input should be metric name like 'churn rate' or 'LTV:CAC'."""
    benchmarks = {
        "churn rate": "Good: <5% monthly, Great: <2% monthly",
        "ltv:cac": "Good: >3, Great: >5",
        "gross margin": "Good: >70%, Great: >80%",
        "net revenue retention": "Good: >100%, Great: >120%",
        "payback period": "Good: <12 months, Great: <6 months"
    }
    
    for key, value in benchmarks.items():
        if metric_name.lower() in key:
            return f"{key}: {value}"
    return "Benchmark not found"

tools = [calculate_metric, lookup_benchmark]

print("LANGCHAIN REACT AGENT")
print("="*60)
print("Tools available:", [t.name for t in tools])

---

## 4. Self-Consistency

### What is Self-Consistency?

Self-Consistency generates **multiple reasoning paths** and takes the **majority vote** on the final answer. This improves accuracy on complex reasoning tasks.

**When to use:**
- High-stakes decisions
- Complex reasoning problems
- When a single answer might be wrong

### Industry Example: Risk Assessment

In [ ]:
def self_consistency(
    prompt: str, 
    n_samples: int = 5, 
    temperature: float = 0.7,
    extract_answer: Callable = None
) -> Dict:
    """
    Self-consistency: Generate multiple samples and take majority vote.
    """
    responses = []
    
    # Generate multiple responses
    for _ in range(n_samples):
        response = model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(temperature=temperature)
        )
        responses.append(response.text)
    
    # Extract answers
    if extract_answer:
        answers = [extract_answer(r) for r in responses]
    else:
        # Default: try to find a clear answer pattern
        answers = []
        for r in responses:
            # Look for common answer patterns
            patterns = [
                r'(?:answer|conclusion|recommendation|verdict)[:\s]+([^.\n]+)',
                r'(?:therefore|thus|hence)[,\s]+([^.\n]+)',
                r'(?:I would|I recommend|The answer is)[:\s]+([^.\n]+)'
            ]
            
            found = False
            for pattern in patterns:
                match = re.search(pattern, r, re.IGNORECASE)
                if match:
                    answers.append(match.group(1).strip().lower())
                    found = True
                    break
            
            if not found:
                # Use last line as answer
                last_line = r.strip().split('\n')[-1]
                answers.append(last_line.strip().lower()[:50])
    
    # Majority vote
    counter = Counter(answers)
    most_common = counter.most_common(1)[0]
    
    return {
        "majority_answer": most_common[0],
        "confidence": most_common[1] / n_samples,
        "vote_distribution": dict(counter),
        "all_responses": responses,
        "all_answers": answers
    }

In [ ]:
# Risk assessment example
risk_question = """
Evaluate this startup investment opportunity:

Company: AI-powered legal document analysis
- Founded 2 years ago
- ARR: $2M (growing 200% YoY)
- Burn rate: $150K/month
- Runway: 18 months
- Team: 15 people, strong technical founders
- Market: Legal tech ($20B market)
- Competition: 3 well-funded competitors
- Asking: $10M at $40M valuation

Should we invest? Analyze step by step, then give your verdict: INVEST, PASS, or NEED MORE INFO.
"""

def extract_verdict(response: str) -> str:
    """Extract investment verdict from response."""
    response_upper = response.upper()
    if "INVEST" in response_upper and "PASS" not in response_upper:
        return "INVEST"
    elif "PASS" in response_upper:
        return "PASS"
    elif "NEED MORE INFO" in response_upper or "MORE INFO" in response_upper:
        return "NEED MORE INFO"
    else:
        return "UNCLEAR"

print("SELF-CONSISTENCY: INVESTMENT DECISION")
print("="*60)

result = self_consistency(
    risk_question,
    n_samples=5,
    temperature=0.8,
    extract_answer=extract_verdict
)

print(f"\n📊 VOTING RESULTS:")
print("-"*40)
for answer, count in result['vote_distribution'].items():
    bar = "█" * (count * 10)
    print(f"  {answer}: {bar} ({count}/{len(result['all_answers'])} = {count/len(result['all_answers']):.0%})")

print(f"\n✅ MAJORITY DECISION: {result['majority_answer']}")
print(f"   Confidence: {result['confidence']:.0%}")

print("\n" + "-"*40)
print("Sample reasoning:")
print(result['all_responses'][0][:500] + "...")

In [ ]:
# Self-consistency for complex math/logic
logic_problem = """
A company offers three pricing tiers:
- Basic: $29/month, max 5 users
- Pro: $79/month, max 20 users
- Enterprise: $199/month, unlimited users

A customer has 18 users and expects to grow to 25 users in 6 months.
They want to minimize total cost over the next 12 months.

What's the optimal purchasing strategy?
Think step by step, calculate the total cost for each option, then recommend.
"""

def extract_recommendation(response: str) -> str:
    """Extract pricing recommendation."""
    response_lower = response.lower()
    if "basic" in response_lower and "enterprise" not in response_lower and "pro" not in response_lower:
        return "basic"
    elif "pro" in response_lower and "then" in response_lower:
        return "pro-then-upgrade"
    elif "enterprise" in response_lower:
        return "enterprise"
    elif "pro" in response_lower:
        return "pro"
    return "unclear"

result = self_consistency(
    logic_problem,
    n_samples=5,
    temperature=0.7,
    extract_answer=extract_recommendation
)

print("SELF-CONSISTENCY: PRICING OPTIMIZATION")
print("="*60)

print(f"\n📊 VOTING RESULTS:")
for answer, count in result['vote_distribution'].items():
    print(f"  {answer}: {count} votes")

print(f"\n✅ RECOMMENDED STRATEGY: {result['majority_answer']}")
print(f"   Confidence: {result['confidence']:.0%}")

---

## 5. Combining Techniques: Production Example

Let's build a comprehensive analysis system that combines multiple techniques.

In [ ]:
class AdvancedAnalysisSystem:
    """
    Production system combining CoT, ToT, and Self-Consistency.
    """
    
    def __init__(self):
        self.model = genai.GenerativeModel('gemini-1.5-flash')
    
    def analyze_with_cot(self, problem: str) -> str:
        """Initial analysis using Chain-of-Thought."""
        prompt = f"""{problem}

Analyze this step by step:
1. Identify the key factors
2. Evaluate each factor
3. Consider interactions between factors
4. Draw conclusions
"""
        response = self.model.generate_content(prompt)
        return response.text
    
    def generate_alternatives(self, problem: str, initial_analysis: str) -> List[str]:
        """Generate alternative perspectives using ToT concept."""
        prompt = f"""Problem: {problem}

Initial Analysis:
{initial_analysis[:500]}

Now, generate 3 alternative perspectives or approaches that challenge or complement this analysis.
Consider: different assumptions, opposing viewpoints, or alternative strategies.
"""
        response = self.model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(temperature=0.8)
        )
        return [response.text]
    
    def validate_with_self_consistency(self, problem: str, recommendation: str) -> Dict:
        """Validate recommendation using self-consistency."""
        prompt = f"""Problem: {problem}

Proposed Recommendation: {recommendation}

Do you agree with this recommendation? Analyze independently and respond with:
AGREE, DISAGREE, or PARTIALLY AGREE

Then explain your reasoning.
"""
        
        def extract_agreement(r):
            r_upper = r.upper()
            if "DISAGREE" in r_upper and "PARTIALLY" not in r_upper:
                return "DISAGREE"
            elif "PARTIALLY" in r_upper:
                return "PARTIALLY AGREE"
            elif "AGREE" in r_upper:
                return "AGREE"
            return "UNCLEAR"
        
        return self_consistency(
            prompt,
            n_samples=3,
            temperature=0.7,
            extract_answer=extract_agreement
        )
    
    def full_analysis(self, problem: str) -> Dict:
        """Run complete analysis pipeline."""
        print("🔍 Starting comprehensive analysis...\n")
        
        # Step 1: CoT analysis
        print("Step 1: Chain-of-Thought analysis...")
        cot_analysis = self.analyze_with_cot(problem)
        
        # Step 2: Generate alternatives
        print("Step 2: Generating alternative perspectives...")
        alternatives = self.generate_alternatives(problem, cot_analysis)
        
        # Step 3: Extract recommendation
        print("Step 3: Extracting recommendation...")
        recommendation_prompt = f"""Based on this analysis:
{cot_analysis[:800]}

Provide a clear, actionable recommendation in one sentence.
"""
        rec_response = self.model.generate_content(recommendation_prompt)
        recommendation = rec_response.text.strip()
        
        # Step 4: Validate with self-consistency
        print("Step 4: Validating with self-consistency...")
        validation = self.validate_with_self_consistency(problem, recommendation)
        
        return {
            "initial_analysis": cot_analysis,
            "alternative_perspectives": alternatives,
            "recommendation": recommendation,
            "validation": validation,
            "confidence": validation["confidence"]
        }

# Test the system
system = AdvancedAnalysisSystem()

business_problem = """
Our B2B SaaS company is facing a decision:

Current state:
- 500 customers, $4M ARR
- 95% of revenue from SMB segment
- Average deal size: $8K/year
- Sales cycle: 2 weeks
- Churn: 5% monthly

Opportunity:
- Enterprise segment interest ($100K+ deals)
- Would require: new sales team, SOC2 compliance, SLAs
- Investment: ~$500K upfront, $200K/year ongoing
- Expected: 10 enterprise deals in year 1

Should we pursue the enterprise segment or double down on SMB?
"""

print("ADVANCED ANALYSIS SYSTEM")
print("="*60)

result = system.full_analysis(business_problem)

print("\n" + "="*60)
print("ANALYSIS RESULTS")
print("="*60)

print("\n📋 Initial Analysis:")
print(result['initial_analysis'][:600] + "...")

print("\n💡 Recommendation:")
print(result['recommendation'])

print("\n✅ Validation:")
print(f"   Consensus: {result['validation']['majority_answer']}")
print(f"   Confidence: {result['validation']['confidence']:.0%}")

---

## Key Takeaways

### Chain-of-Thought (CoT)
- Add "Let's think step by step" for instant improvement
- Few-shot CoT with examples is more powerful
- Best for: math, logic, multi-step reasoning

### Tree-of-Thought (ToT)
- Generate multiple approaches
- Evaluate and prune
- Expand the most promising
- Best for: complex decisions, creative problems

### ReAct
- Interleave reasoning and action
- Foundation for tool-using agents
- Best for: real-world tasks requiring tools

### Self-Consistency
- Multiple samples + majority vote
- Increases reliability on complex tasks
- Best for: high-stakes decisions

---

## Next Steps

Proceed to **Lesson 2.3: Prompt Security** to learn:
- Prompt injection attacks
- Defense strategies
- Security testing